## Tensorflow aur Keras ka Introduction

> Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai. Poore learning material ke liye repository par jaiye.

### Neural Frameworks

Humne seekha hai ki neural network ko train karne ke liye aapko chahiye:
* Matrices (tensors) ko tezi se multiply karna
* Gradients calculate karna taaki gradient descent optimization ho sake

Neural network frameworks aapko yeh karne dete hain:
* Tensors ke saath us compute par kaam karna jo available ho, chahe woh CPU ho, GPU ho, ya even TPU bhi
* Gradients ko automatically calculate karna (yeh sab built-in tensor functions ke liye explicitly program kiye gaye hain)

Optionally:
* Neural Network constructors / higher-level API (network ko layers ke sequence ke roop mein describe karna)
* Simple training functions (`fit`, jaisa Scikit Learn mein hai)
* Gradient descent ke alawa kai optimization algorithms
* Data handling abstractions (jo ideally GPU par bhi kaam karein)

### Sabse Popular Frameworks

* Tensorflow 1.x - pehla widely available framework (Google). Yeh static computation graph define karne, use GPU par bhejne aur explicitly evaluate karne ki permission deta tha
* PyTorch - Facebook ka framework jiski popularity badh rahi hai
* Keras - Tensorflow/PyTorch ke upar high-level API, jo neural networks ka use unified aur simple banata hai (Francois Chollet)
* Tensorflow 2.x + Keras - Tensorflow ka naya version jisme Keras ki functionality integrated hai, jo **dynamic computation graph** ko support karta hai, jisse numpy (aur PyTorch) jaise tensor operations karna possible hota hai

Hum Tensorflow 2.x aur Keras par focus karenge. Make sure kijiye ki aapke paas Tensorflow ka version 2.x.x installed hai:
```
pip install tensorflow
```
ya
```
conda install tensorflow
```

In [ ]:
import tensorflow as tf
import numpy as np
print(tf.__version__)

## Basic Concepts: Tensor

**Tensor** ek multi-dimensional array hai. Alag-alag types ke data ko represent karne ke liye tensors ka use karna bahut convenient hai:
* 400x400 - black-and-white picture
* 400x400x3 - color picture
* 16x400x400x3 - 16 color pictures ka minibatch
* 25x400x400x3 - 25-frames per second video ka ek second
* 8x25x400x400x3 - 8 1-second videos ka minibatch

### Simple Tensors

Aap np-arrays ki lists se aasaani se simple tensors bana sakte hain, ya random tensors generate kar sakte hain:

In [ ]:
a = tf.constant([[1,2],[3,4]])
print(a)
a = tf.random.normal(shape=(10,3))
print(a)

Aap tensors par arithmetic operations kar sakte hain, jo element-wise hote hain, jaise numpy mein. Tensors ko zaroorat padne par required dimension tak automatically expand kiya jaata hai. Tensor se numpy-array nikalne ke liye, `.numpy()` use kijiye:

In [ ]:
print(a-a[0])
print(tf.exp(a)[0].numpy())

## Variables

Variables ka use tensor values ko represent karne ke liye kiya jaata hai jinhe `assign` aur `assign_add` use karke modify kiya ja sakta hai. Inhe aksar neural network ke weights ko represent karne ke liye use kiya jaata hai.

Example ke taur par, yahan tensor `a` ki saari rows ka sum paane ka ek simple tareeka diya gaya hai:

In [ ]:
s = tf.Variable(tf.zeros_like(a[0]))
for i in a:
  s.assign_add(i)

print(s)

In [ ]:
tf.reduce_sum(a,axis=0)

## Gradients Calculate karna

Back propagation ke liye, aapko gradients calculate karne honge. Yeh `tf.GradientTape()` method use karke kiya jaata hai:
 * Hamari calculations ke aas-paas `with tf.GradientTape` block add karein
 * Un tensors ko mark karein jinke respect mein hamein gradients calculate karne hain, `tape.watch` call karke (saare variables automatically watch hote hain)
 * Jo bhi zaroori ho woh compute karein (computational graph banayein)
 * `tape.gradient` use karke gradients paayein

In [ ]:
a = tf.random.normal(shape=(2, 2))
b = tf.random.normal(shape=(2, 2))

with tf.GradientTape() as tape:
  tape.watch(a)  # Start recording the history of operations applied to `a`
  c = tf.sqrt(tf.square(a) + tf.square(b))  # Do some math using `a`
  # What's the gradient of `c` with respect to `a`?
  dc_da = tape.gradient(c, a)
  print(dc_da)

## Example 1: Linear Regression

Ab hum **linear regression** ki traditional problem solve karne ke liye kaafi jaante hain. Chaliye ek chhota synthetic dataset banate hain:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
import random

In [ ]:
np.random.seed(13) # pick the seed for reproducability - change it to explore the effects of random variations

train_x = np.linspace(0, 3, 120)
train_labels = 2 * train_x + 0.9 + np.random.randn(*train_x.shape) * 0.5

plt.scatter(train_x,train_labels)

Linear Regression ko ek straight line $f_{W,b}(x) = Wx+b$ se define kiya jaata hai, jahan $W, b$ model parameters hain jinhe hamein dhoondhna hai. Hamare dataset $\{x_i,y_u\}_{i=1}^N$ par error (jise **loss function** bhi kehte hain) ko mean square error ke roop mein define kiya ja sakta hai:
$$
\mathcal{L}(W,b) = {1\over N}\sum_{i=1}^N (f_{W,b}(x_i)-y_i)^2
$$

Aaiye apna model aur loss function define karte hain:

In [ ]:
input_dim = 1
output_dim = 1
learning_rate = 0.1

# This is our weight matrix
w = tf.Variable([[100.0]])
# This is our bias vector
b = tf.Variable(tf.zeros(shape=(output_dim,)))

def f(x):
  return tf.matmul(x,w) + b

def compute_loss(labels, predictions):
  return tf.reduce_mean(tf.square(labels - predictions))

Hum model ko minibatches ki ek series par train karenge. Hum gradient descent use karenge, aur neeche diye formulas se model ke parameters adjust karenge:
$$
\begin{array}{l}
W^{(n+1)}=W^{(n)}-\eta\frac{\partial\mathcal{L}}{\partial W} \\
b^{(n+1)}=b^{(n)}-\eta\frac{\partial\mathcal{L}}{\partial b} \\
\end{array}
$$

In [ ]:
def train_on_batch(x, y):
  with tf.GradientTape() as tape:
    predictions = f(x)
    loss = compute_loss(y, predictions)
    # Note that `tape.gradient` works with a list as well (w, b).
    dloss_dw, dloss_db = tape.gradient(loss, [w, b])
  w.assign_sub(learning_rate * dloss_dw)
  b.assign_sub(learning_rate * dloss_db)
  return loss

Chalo training karte hain. Hum dataset se kai baar (jise **epochs** kehte hain) guzrenge, use minibatches mein divide karenge aur upar define kiye gaye function ko call karenge:

In [ ]:
# Shuffle the data.
indices = np.random.permutation(len(train_x))
features = tf.constant(train_x[indices],dtype=tf.float32)
labels = tf.constant(train_labels[indices],dtype=tf.float32)

In [ ]:
batch_size = 4
for epoch in range(10):
  for i in range(0,len(features),batch_size):
    loss = train_on_batch(tf.reshape(features[i:i+batch_size],(-1,1)),tf.reshape(labels[i:i+batch_size],(-1,1)))
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

Humne ab optimized parameters $W$ aur $b$ paa liye hain. Dhyan dijiye ki unki values us original value ke similar hain jo dataset banate waqt use ki gayi thi ($W=2, b=1$).

In [ ]:
w,b

In [ ]:
plt.scatter(train_x,train_labels)
x = np.array([min(train_x),max(train_x)])
y = w.numpy()[0,0]*x+b.numpy()[0]
plt.plot(x,y,color='red')

## Computational Graph aur GPU Computations

Jab bhi hum tensor expression compute karte hain, Tensorflow ek computational graph banata hai jise available computing devices, jaise CPU ya GPU, par chalaya ja sakta hai. Kyunki hum apne code mein arbitrary Python functions use kar rahe the, woh computational graph ka hissa nahi ban sakte, aur isliye jab hum apna code GPU par chalate hain, toh hamein CPU aur GPU ke beech data baar-baar pass karna hoga aur custom function ko CPU par calculate karna hoga.

Tensorflow hamein `@tf.function` decorator use karke apne Python function ko mark karne deta hai, jo is function ko usi computational graph ka hissa bana dega. Yeh decorator un functions par apply kiya ja sakta hai jo standard Tensorflow tensor operations use karte hain.

In [ ]:
@tf.function
def train_on_batch(x, y):
  with tf.GradientTape() as tape:
    predictions = f(x)
    loss = compute_loss(y, predictions)
    # Note that `tape.gradient` works with a list as well (w, b).
    dloss_dw, dloss_db = tape.gradient(loss, [w, b])
  w.assign_sub(learning_rate * dloss_dw)
  b.assign_sub(learning_rate * dloss_db)
  return loss

Code mein koi change nahi hua hai, lekin agar aap is code ko GPU par aur bade dataset par chala rahe hote - toh aapko speed mein difference mehsoos hota.

## Dataset API

Tensorflow mein data ke saath kaam karne ke liye ek convenient API hai. Chaliye ise use karke dekhte hain. Hum apne model ko scratch se train bhi karenge.

In [ ]:
w.assign([[10.0]])
b.assign([0.0])

# Create a tf.data.Dataset object for easy batched iteration
dataset = tf.data.Dataset.from_tensor_slices((train_x.astype(np.float32), train_labels.astype(np.float32)))
dataset = dataset.shuffle(buffer_size=1024).batch(256)

for epoch in range(10):
  for step, (x, y) in enumerate(dataset):
    loss = train_on_batch(tf.reshape(x,(-1,1)), tf.reshape(y,(-1,1)))
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

## Example 2: Classification

Ab hum binary classification problem par vichaar karenge. Is type ki problem ka ek achha example hoga tumor ko classify karna, yeh decide karna ki woh malignant hai ya benign, uske size aur age ke basis par.

Core model regression jaisa hi hai, lekin hamein alag loss function use karna hoga. Chaliye sample data taiyaar karke shuru karte hain:

In [ ]:
np.random.seed(0) # pick the seed for reproducibility - change it to explore the effects of random variations

n = 100
X, Y = make_classification(n_samples = n, n_features=2,
                           n_redundant=0, n_informative=2, flip_y=0.05,class_sep=1.5)
X = X.astype(np.float32)
Y = Y.astype(np.int32)

split = [ 70*n//100, (15+70)*n//100 ]
train_x, valid_x, test_x = np.split(X, split)
train_labels, valid_labels, test_labels = np.split(Y, split)

In [ ]:
def plot_dataset(features, labels, W=None, b=None):
    # prepare the plot
    fig, ax = plt.subplots(1, 1)
    ax.set_xlabel('$x_i[0]$ -- (feature 1)')
    ax.set_ylabel('$x_i[1]$ -- (feature 2)')
    colors = ['r' if l else 'b' for l in labels]
    ax.scatter(features[:, 0], features[:, 1], marker='o', c=colors, s=100, alpha = 0.5)
    if W is not None:
        min_x = min(features[:,0])
        max_x = max(features[:,1])
        min_y = min(features[:,1])*(1-.1)
        max_y = max(features[:,1])*(1+.1)
        cx = np.array([min_x,max_x],dtype=np.float32)
        cy = (0.5-W[0]*cx-b)/W[1]
        ax.plot(cx,cy,'g')
        ax.set_ylim(min_y,max_y)
    fig.show()

In [ ]:
plot_dataset(train_x, train_labels)

## Data ko Normalize karna

Training se pehle, yeh common practice hai ki hamare input features ko standard range [0,1] (ya [-1,1]) mein laaya jaye. Iske peeche ke exact reasons par hum course mein baad mein baat karenge, lekin sankshep mein reason yeh hai. Hum chahte hain ki hamare network se flow hone wali values na bahut badi hon na bahut chhoti, aur aam taur par hum agree karte hain ki saari values chhote range mein, 0 ke kareeb rahein. Isliye hum weights ko chhote random numbers se initialize karte hain aur signals ko usi range mein banaye rakhte hain.

Data ko normalize karte waqt, hamein minimum value ghatani hoti hai aur range se divide karna hota hai. Hum training data use karke minimum value aur range calculate karte hain, aur phir test/validation datasets ko training set se mile usi min/range values se normalize karte hain. Aisa isliye hai kyunki real life mein hamein sirf training set ka hi pata hoga, aur network se prediction karwane ke liye aane wali saari naye values ka nahi pata hoga. Kabhi-kabhi, nayi value [0,1] range ke bahar bhi ho sakti hai, lekin yeh bahut important nahi hai.

In [ ]:
train_x_norm = (train_x-np.min(train_x)) / (np.max(train_x)-np.min(train_x))
valid_x_norm = (valid_x-np.min(train_x)) / (np.max(train_x)-np.min(train_x))
test_x_norm = (test_x-np.min(train_x)) / (np.max(train_x)-np.min(train_x))

## Single-Layer Perceptron ki Training

Chaliye Tensorflow ke gradient computing mechanism ka use karke ek single-layer perceptron ko train karte hain.

Hamare neural network mein 2 inputs aur 1 output hoga. Weight matrix $W$ ka size $2\times1$ hoga, aur bias vector $b$ -- $1$.

Core model pehle example jaisa hi hoga, lekin loss function logistic loss hoga. Logistic loss implement karne ke liye, hamein apne network ke output ke roop mein **probability** ki value paani hogi, yaani hamein output $z$ ko `sigmoid` activation function use karke [0,1] ki range mein laana hoga: $p=\sigma(z)$.

Agar hamein i-th input value ke liye probability $p_i$ milti hai, jo actual class $y_i\in\{0,1\}$ se related hai, toh hum loss ko is tarah calculate karte hain: $\mathcal{L_i}=-(y_i\log p_i + (1-y_i)log(1-p_i))$.

Tensorflow mein, in dono steps (sigmoid apply karna aur phir logistic loss) ko `sigmoid_cross_entropy_with_logits` function ke ek call se kiya ja sakta hai. Kyunki hum apne network ko minibatches mein train kar rahe hain, hamein minibatch ke saare elements ke loss ko `reduce_mean` use karke average karna hoga:

In [ ]:
W = tf.Variable(tf.random.normal(shape=(2,1)),dtype=tf.float32)
b = tf.Variable(tf.zeros(shape=(1,),dtype=tf.float32))

learning_rate = 0.1

@tf.function
def train_on_batch(x, y):
  with tf.GradientTape() as tape:
    z = tf.matmul(x, W) + b
    loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=z))
  dloss_dw, dloss_db = tape.gradient(loss, [W, b])
  W.assign_sub(learning_rate * dloss_dw)
  b.assign_sub(learning_rate * dloss_db)
  return loss

Hum 16 elements ka minibatch use karenge, aur training ke kuch epochs karenge:

In [ ]:
# Create a tf.data.Dataset object for easy batched iteration
dataset = tf.data.Dataset.from_tensor_slices((train_x_norm.astype(np.float32), train_labels.astype(np.float32)))
dataset = dataset.shuffle(128).batch(2)

for epoch in range(10):
  for step, (x, y) in enumerate(dataset):
    loss = train_on_batch(x, tf.expand_dims(y,1))
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

In [ ]:
plot_dataset(train_x,train_labels,W.numpy(),b.numpy())

Dekhte hain ki hamara model validation data par kaisa behave karta hai.

In [ ]:
pred = tf.matmul(test_x,W)+b
fig,ax = plt.subplots(1,2)
ax[0].scatter(test_x[:,0],test_x[:,1],c=pred[:,0]>0.5)
ax[1].scatter(test_x[:,0],test_x[:,1],c=valid_labels)

Validation data par accuracy calculate karne ke liye, hum boolean type ko float mein convert kar sakte hain aur average calculate kar sakte hain:

In [ ]:
tf.reduce_mean(tf.cast(((pred[0]>0.5)==test_labels),tf.float32))

Samajhte hain ki yahan kya ho raha hai:
* `pred` network dwara predict ki gayi values hain. Yeh strictly probabilities nahi hain, kyunki humne koi activation function use nahi kiya hai, lekin 0.5 se badi values class 1 ke liye hain, aur chhoti values class 0 ke liye.
* `pred[0]>0.5` ek boolean tensor banata hai, jahan `True` class 1 ko darshata hai, aur `False` class 0 ko.
* Hum is tensor ko expected labels `valid_labels` se compare karte hain, jisse correct predictions ka boolean vector milta hai, jahan `True` sahi prediction ko darshata hai, aur `False` galat ko.
* Hum is tensor ko `tf.cast` use karke floating point mein convert karte hain.
* Phir hum `tf.reduce_mean` use karke mean value nikalte hain - yahi hamari desired accuracy hai.

## TensorFlow/Keras Optimizers ka Use

TensorFlow Keras ke saath gehraai se jud hua hai, jisme kai useful features shaamil hain. Example ke liye, hum alag-alag **optimization algorithms** use kar sakte hain. Chaliye aisa karte hain aur training ke dauran mili accuracy bhi print karte hain.

In [ ]:
optimizer = tf.keras.optimizers.Adam(0.01)

W = tf.Variable(tf.random.normal(shape=(2,1)))
b = tf.Variable(tf.zeros(shape=(1,),dtype=tf.float32))

@tf.function
def train_on_batch(x, y):
  vars = [W, b]
  with tf.GradientTape() as tape:
    z = tf.sigmoid(tf.matmul(x, W) + b)
    loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(z,y))
    correct_prediction = tf.equal(tf.round(y), tf.round(z))
    acc = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    grads = tape.gradient(loss, vars)
    optimizer.apply_gradients(zip(grads,vars))
  return loss,acc

for epoch in range(20):
  for step, (x, y) in enumerate(dataset):
    loss,acc = train_on_batch(tf.reshape(x,(-1,2)), tf.reshape(y,(-1,1)))
  print('Epoch %d: last batch loss = %.4f, acc = %.4f' % (epoch, float(loss),acc))

**Task 1**: Training ke dauran training aur validation data par loss function aur accuracy ke graphs plot kijiye

**Task 2**: Is code ka use karke MNIST classification problem solve karne ki koshish kijiye. Hint: `softmax_crossentropy_with_logits` ya `sparse_softmax_cross_entropy_with_logits` ko loss function ke roop mein use kijiye. Pehle case mein aapko expected output values ko *one-hot encoding* mein dena hoga, aur doosre case mein - integer class number ke roop mein.

## Keras
### Deep Learning for Humans

* Keras ek library hai jise originally Francois Chollet ne Tensorflow, CNTK aur Theano ke upar kaam karne ke liye develop kiya tha, taaki saare lower-level frameworks ko unify kiya ja sake. Aap abhi bhi Keras ko ek alag library ke roop mein install kar sakte hain, lekin aisa karna recommended nahi hai.
* Ab Keras Tensorflow library ka hissa hai.
* Aap aasaani se layers se neural networks bana sakte hain.
* Isme `fit` function hai jo saari training karta hai, saath hi common data (jaise pictures, text waghera) ke saath kaam karne ke liye kai functions available hain.
* Bahut saare samples available hain.
* Functional API vs Sequential API

Keras neural networks ke liye high-level abstractions provide karta hai, jisse hum tensors aur gradients ke bajay layers, models aur optimizers ke terms mein kaam kar sakte hain.

Keras ke creator ki famous deep learning book: [Deep Learning with Python](https://www.manning.com/books/deep-learning-with-python)

### Functional API

Jab hum functional API use karte hain, toh hum network ke **input** ko `keras.Input` ke roop mein define karte hain, aur phir use kai computations se pass karke **output** calculate karte hain. Aakhir mein, hum **model** ko ek object ke roop mein define karte hain jo input ko output mein transform karta hai.

Ek baar jab hamein **model** object mil jaata hai, toh hamein ise:
* **Compile karna hota hai**, jisme hamein loss function aur optimizer specify karna hota hai jo hum apne model ke saath use karna chahte hain.
* **Train karna hota hai** `fit` function ko training (aur possibly validation) data ke saath call karke.

In [ ]:
inputs = tf.keras.Input(shape=(2,))
z = tf.keras.layers.Dense(1,kernel_initializer='glorot_uniform',activation='sigmoid')(inputs)
model = tf.keras.models.Model(inputs,z)

model.compile(tf.keras.optimizers.Adam(0.1),'binary_crossentropy',['accuracy'])
model.summary()
h = model.fit(train_x_norm,train_labels,batch_size=8,epochs=15)

In [ ]:
plt.plot(h.history['accuracy'])

### Sequential API

Alternatively, hum model ko **layers ke sequence** ke roop mein soch sakte hain, aur un layers ko `model` object mein add karke hi specify kar sakte hain:

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(5,activation='sigmoid',input_shape=(2,)))
model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

model.compile(tf.keras.optimizers.Adam(0.1),'binary_crossentropy',['accuracy'])
model.summary()
model.fit(train_x_norm,train_labels,validation_data=(test_x_norm,test_labels),batch_size=8,epochs=15)

## Classification Loss Functions

Network ki final layer par loss function aur activation function ko sahi tarah se specify karna important hai. Main rules yeh hain:
* Agar network ka ek output hai (**binary classification**), toh hum **sigmoid** activation function use karte hain, aur **multiclass classification** ke liye - **softmax**.
* Agar output class one-hot-encoding ke roop mein represent ki gayi hai, toh loss function **cross entropy loss** (categorical cross-entropy) hoga. Agar output mein class number hai - **sparse categorical cross-entropy**. **Binary classification** ke liye - **binary cross-entropy** (jo **log loss** jaisa hi hai) use kijiye.
* **Multi-label classification** woh situation hai jab koi object ek hi time par kai classes se belong kar sakta hai. Is case mein, hamein labels ko one-hot encoding use karke encode karna hoga aur **sigmoid** ko activation function ke roop mein use karna hoga, taaki har class ki probability 0 aur 1 ke beech ho.

| Classification | Label Format | Activation Function | Loss |
|---------------|-----------------------|-----------------|----------|
| Binary      | Probability of 1st class | sigmoid | binary crossentropy |
| Binary      | One-hot encoding (2 outputs) | softmax | categorical crossentropy |
| Multiclass |  One-hot encoding | softmax | categorical crossentropy |
| Multiclass | Class Number | softmax | sparse categorical crossentropy |
| Multilabel | One-hot encoding | sigmoid | categorical crossentropy |

> Binary classification ko multi-class classification ke special case ke roop mein bhi handle kiya ja sakta hai jisme do outputs hote hain. Is case mein, hamein **softmax** use karna hoga.

**Task 3**:
Keras use karke MNIST classifier ko train kijiye:
* Dhyan dijiye ki Keras mein kuch standard datasets shaamil hain, jinme MNIST bhi hai. Keras se MNIST use karne ke liye, aapko sirf kuch lines ke code ki zaroorat hogi (zyada jaankari [yahan](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist))
* Alag-alag network configurations try kijiye, jaise layers/neurons ki sankhya, activation functions.

Aapne sabse best accuracy kya paayi?

## Key Takeaways

* Tensorflow aapko tensors par low level par kaam karne deta hai, jisse aapko maximum flexibility milti hai.
* Data (`td.Data`) aur layers (`tf.layers`) ke saath kaam karne ke liye convenient tools available hain.
* Beginners ya common tasks ke liye, **Keras** use karne ki recommendation hai, jo layers se network banane deta hai.
* Agar non-standard architectures ki zaroorat ho, toh aap apna khud ka Keras layer bana sakte hain aur phir use Keras models mein use kar sakte hain.
* PyTorch ko bhi dekhna aur approaches ko compare karna ek achha idea hai.

Keras aur Tensorflow 2.0 par Keras ke creator dwara ek achhi sample notebook [yahan](https://t.co/k694J95PI8) mil sakti hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->